# UNIDAD I: Introducción a algoritmos y estructuras de datos
## 🔀 Capitulo 2 DSAP: OOP — Herencia y Encapsulamiento
### Programación III - Lic. en Sistemas

![Coding](images/pexels-ketut-subiyanto-4933843.jpg)

[Foto de Ketut Subiyanto](https://www.pexels.com/es-es/foto/hombre-amor-jardin-jugando-4933843/)
---
> **Prerequisito:** Notebook 1/3 — Clases y Objetos (§2.1–2.3)  
> La herencia es la forma en que Python permite **reutilizar y especializar** clases existentes sin duplicar código.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap02/02_Herencia_Encapsulamiento_Teoria.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook serás capaz de:

1. **Definir** una clase derivada usando `class Hijo(Padre):` y comprender la cadena de herencia
2. **Usar** `super()` correctamente para llamar al constructor y a métodos de la clase base
3. **Sobreescribir** (*override*) métodos de la clase base y agregar comportamiento especializado
4. **Explicar** el *Method Resolution Order* (MRO) y el algoritmo C3 de linearización
5. **Implementar** encapsulamiento con `_atributo` / `__atributo` y propiedades con `@property` / `@setter`

## 📑 Tabla de Contenidos

| Sección | Tema | Tiempo estimado |
|---------|------|----------------|
| **§1** | Herencia: base, derivada, `super()`, *override* | 35 min |
| **§2** | *Method Resolution Order* (MRO) y problema del diamante | 20 min |
| **§3** | Encapsulamiento: name mangling, `@property`, `@setter` | 25 min |
| 🧪 | Tests unitarios automatizados | 10 min |
| 📋 | Resumen, autoevaluación, referencias | 10 min |
| **Total** | | **~100 min** |

---

### 🧩 Analogía: extensión de un plano arquitectónico

En el Notebook 1 vimos que una clase es como el **plano de un edificio**. Ahora, la herencia es como tomar ese plano y crear una **versión especializada** — conservas todo lo que funciona y sólo modificas o agregás lo que necesitás.

```text
"Un edificio de oficinas hereda todas las características de un edificio genérico
 (cimientos, paredes, techo) y agrega: sala de reuniones, acceso con tarjeta, etc."
```

En el libro, el ejemplo central es la jerarquía `Progresion`:

```text
Progresion          <- clase base (genera secuencias)
  ProgresionAritmetica   <- suma un incremento fijo
  ProgresionGeometrica   <- multiplica por una razón fija
  ProgresionFibonacci    <- acumula los dos anteriores
```

## ⚙️ Configuración

In [ ]:
import sys
import unittest
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Verificar versión de Python
assert sys.version_info >= (3, 8), f"Python 3.8+ requerido. Versión actual: {sys.version}"
print(f"Python {sys.version}")
print("Módulos cargados: sys, unittest, matplotlib")
print("Listo para explorar §2.4–2.5 Herencia y Encapsulamiento")

---
## 🧬 Sección 1 — Herencia: Base, Derivada y `super()`

### Vocabulario esencial

| Término | Sinónimos | Descripción |
|---------|-----------|-------------|
| **Clase base** | Superclase, clase padre | Define el comportamiento general |
| **Clase derivada** | Subclase, clase hija | Hereda y especializa la base |
| **Herencia** | *inheritance* | Mecanismo para reutilizar y extender |
| **Override** | Sobreescritura | Redefinir un método en la subclase |
| **`super()`** | — | Proxy a la clase inmediatamente superior en el MRO |

### Sintaxis Python

```python
class Base:
    def __init__(self, x):
        self._x = x

    def describir(self):
        return f"Base con x={self._x}"


class Derivada(Base):              # Derivada hereda de Base
    def __init__(self, x, y):
        super().__init__(x)        # llama al __init__ de Base
        self._y = y                # agrega estado propio

    def describir(self):           # override del método de Base
        base_desc = super().describir()     # reutiliza la lógica de Base
        return f"{base_desc}, Derivada con y={self._y}"
```

### Lo que hereda la clase derivada

1. **Todos los métodos** (salvo que se sobreescriban)
2. **Todos los atributos de clase** de la base  
3. **El constructor**, si no se define uno propio
4. **Métodos especiales** (`__str__`, `__repr__`, etc.)

> 💡 La subclase puede **agregar** nuevos métodos/atributos, **sobreescribir** (modificar) existentes, o **extender** llamando a `super()`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ejemplo de introducción: jerarquía TarjetaCredito -> TarjetaPredatoria
# Basado en §2.4 del libro (CreditCard -> PredatoryCreditCard)
# ─────────────────────────────────────────────────────────────────────────────

class TarjetaCredito:
    """Clase base: tarjeta de crédito simple (del Notebook 1)."""

    def __init__(self, cliente, banco, cuenta, limite):
        self._cliente = cliente
        self._banco   = banco
        self._cuenta  = cuenta
        self._limite  = float(limite)
        self._saldo   = 0.0

    @property
    def cliente(self): return self._cliente
    @property
    def saldo(self):   return self._saldo
    @property
    def limite(self):  return self._limite

    def cargar(self, precio):
        if precio <= 0:
            raise ValueError("Precio debe ser positivo")
        if self._saldo + precio > self._limite:
            return False
        self._saldo += precio
        return True

    def pagar(self, monto):
        if monto <= 0:
            raise ValueError("Monto debe ser positivo")
        self._saldo = max(0.0, self._saldo - monto)

    def __str__(self):
        return f"[{self._cuenta[-4:]}] {self._cliente} | Saldo: ${self._saldo:.2f}"

    def __repr__(self):
        return f"TarjetaCredito({self._cliente!r}, {self._banco!r}, {self._cuenta!r}, {self._limite})"


class TarjetaPredatoria(TarjetaCredito):
    """
    Subclase que agrega:
      - Tasa de interés mensual
      - Cargo adicional cuando se rechaza un cobro (sobre_limite_cargo)
      - Override de __str__ y __repr__
    Basado en §2.4 del libro (PredatoryCreditCard).
    """

    def __init__(self, cliente, banco, cuenta, limite, tasa):
        """
        Args:
            tasa: tasa de interés mensual (%), ej. 0.0125 -> 1.25% mensual
        """
        super().__init__(cliente, banco, cuenta, limite)  # delegar a la base
        self._tasa = float(tasa)

    @property
    def tasa(self): return self._tasa

    def cargar(self, precio):
        """Override: aplica cargo adicional si el crédito es rechazado."""
        aprobado = super().cargar(precio)   # reutiliza la lógica de la base
        if not aprobado:
            # penalidad por intento fallido
            self._saldo = min(self._saldo + 5, self._limite)
        return aprobado

    def cobrar_interes(self):
        """Aplica la tasa de interés mensual al saldo actual."""
        if self._saldo > 0:
            self._saldo *= (1 + self._tasa)

    def __str__(self):
        base = super().__str__()
        return f"{base} [Predatoria | Tasa: {self._tasa*100:.2f}%/mes]"

    def __repr__(self):
        return (f"TarjetaPredatoria({self._cliente!r}, {self._banco!r}, "
                f"{self._cuenta!r}, {self._limite}, {self._tasa})")


# ── Demo ─────────────────────────────────────────────────────────────────────
simple    = TarjetaCredito("Ana", "Banco A", "1111-2222", 1000)
predatoria = TarjetaPredatoria("Bob", "TurboBank", "3333-4444", 500, 0.0125)

# isinstance: verificación de tipo en jerarquía
print(f"simple     es TarjetaCredito:   {isinstance(simple, TarjetaCredito)}")
print(f"predatoria es TarjetaCredito:   {isinstance(predatoria, TarjetaCredito)}")
print(f"predatoria es TarjetaPredatoria:{isinstance(predatoria, TarjetaPredatoria)}")
print(f"simple     es TarjetaPredatoria:{isinstance(simple, TarjetaPredatoria)}")

# uso
predatoria.cargar(400)
print(f"\nDespués de cargar $400: {predatoria}")
predatoria.cargar(200)      # intento fallido → penalidad $5
print(f"Intento fallido $200:   {predatoria}")
predatoria.cobrar_interes()
print(f"Después de interés:     {predatoria}")

print(f"\nrepr: {predatoria!r}")

# issubclass: verificación de clases
print(f"\nTarjetaPredatoria es subclase de TarjetaCredito: "
      f"{issubclass(TarjetaPredatoria, TarjetaCredito)}")

### 🔍 Análisis: ¿Qué hace `super()` exactamente?

`super()` retorna un **objeto proxy** que delega la llamada al método al **próximo en el MRO** (Method Resolution Order). En Python 3, `super()` sin argumentos dentro de un método ya sabe en qué clase está y qué instancia se usa.

```text
TarjetaPredatoria.cargar(self, precio)
  └─ super().cargar(precio)         <- delega a TarjetaCredito.cargar
       └─ lógica de la clase base   <- sin duplicar código
```

### Tres escenarios de override

| Escenario | Código | Cuándo usarlo |
|-----------|--------|--------------|
| **Reemplazar** | `def metodo(self): ...` (sin `super()`) | El comportamiento cambia completamente |
| **Extender** | `super().metodo()` al inicio | Se agrega funcionalidad después de la base |
| **Decorar** | `super().metodo()` en el medio o al final | Se añade antes y después de la base |

> ⚠️ Siempre llamar `super().__init__()` en subclases para no "romper" la inicialización de la clase base.

### 🏗️ Ejemplo central del libro: jerarquía `Progresion`

El libro (§2.4) presenta una jerarquía para generar secuencias numéricas. Esta jerarquía es un modelo perfecto de herencia:

```text
Progresion                    <- clase base
  siguiente_valor()            <- gancho (hook) que cada subclase redefine
  imprimir(n)                  <- método que usa el gancho (definido una sola vez)

ProgresionAritmetica(Progresion)   <- a[i] = a[0] + i * incremento
ProgresionGeometrica(Progresion)   <- a[i] = a[0] * base^i
ProgresionFibonacci(Progresion)    <- a[i] = a[i-1] + a[i-2]
```

Este patrón se llama **Template Method** — la clase base define el esqueleto del algoritmo y las subclases rellenan los pasos específicos. Lo veremos formalmente con ABCs en el Notebook 3.

In [ ]:
class Progresion:
    """
    Clase base para progresiones numéricas (§2.4 del libro).
    
    Implementa el protocolo de iterador de Python:
      __iter__ + __next__  -> la instancia misma es el iterador
    La subclases sólo deben sobreescribir _avanzar().
    """

    def __init__(self, inicio=0):
        self._actual = inicio

    def _avanzar(self):
        """
        Gancho (hook): actualiza self._actual al siguiente valor.
        Las subclases DEBEN sobreescribir este método.
        (En Python puro, sin ABCs; en Notebook 3 lo haremos abstracto.)
        """
        self._actual += 1          # comportamiento por defecto: secuencia 0,1,2,...

    def __next__(self):
        """Retorna el valor actual y avanza al próximo."""
        if self._actual is None:
            raise StopIteration
        respuesta = self._actual
        self._avanzar()
        return respuesta

    def __iter__(self):
        """La instancia es su propio iterador."""
        return self

    def imprimir(self, n: int) -> None:
        """Imprime los primeros n valores de la progresión."""
        valores = [next(self) for _ in range(n)]
        print(', '.join(map(str, valores)))

    def tomar(self, n: int) -> list:
        """Retorna los primeros n valores como lista (sin modificar estado)."""
        return [next(self) for _ in range(n)]


class ProgresionAritmetica(Progresion):
    """
    Progresión aritmética: a[i] = inicio + i * incremento.
    
    Ejemplo: inicio=2, incremento=3  ->  2, 5, 8, 11, 14, ...
    """

    def __init__(self, incremento=1, inicio=0):
        super().__init__(inicio)
        self._incremento = incremento

    def _avanzar(self):                    # override del gancho
        self._actual += self._incremento


class ProgresionGeometrica(Progresion):
    """
    Progresión geométrica: a[i] = inicio * base^i.
    
    Ejemplo: inicio=1, base=2  ->  1, 2, 4, 8, 16, ...
    """

    def __init__(self, base=2, inicio=1):
        super().__init__(inicio)
        self._base = base

    def _avanzar(self):
        self._actual *= self._base


class ProgresionFibonacci(Progresion):
    """
    Progresión de Fibonacci: a[i] = a[i-1] + a[i-2].
    
    Requiere llevar dos valores previos. Override completo de _avanzar.
    """

    def __init__(self, primero=0, segundo=1):
        super().__init__(primero)
        self._prev = segundo - primero    # truco del libro para el primer paso

    def _avanzar(self):
        self._prev, self._actual = self._actual, self._prev + self._actual


# ── Demo ─────────────────────────────────────────────────────────────────────
print("Secuencia base (0, 1, 2, ...):")
Progresion(0).imprimir(8)

print("\nAritmética (inicio=5, incremento=4):")
ProgresionAritmetica(incremento=4, inicio=5).imprimir(8)

print("\nGeométrica (inicio=1, base=3):")
ProgresionGeometrica(base=3, inicio=1).imprimir(8)

print("\nFibonacci (0, 1, 1, 2, 3, ...):")
ProgresionFibonacci(0, 1).imprimir(10)

print("\nFibonacci (4, 6, 10, 16, ...) con inicio distinto:")
ProgresionFibonacci(4, 6).imprimir(8)

# Herencia confirmada
pa = ProgresionAritmetica(2, 0)
print(f"\nisisntance(pa, Progresion):           {isinstance(pa, Progresion)}")
print(f"issubclass(ProgresionFibonacci, Progresion): {issubclass(ProgresionFibonacci, Progresion)}")

# Uso como iterador real
pa10 = ProgresionAritmetica(10, 0)
suma = sum(next(pa10) for _ in range(5))   # 0+10+20+30+40 = 100
print(f"\nSuma de primeros 5 términos (aritmética 0,10,20,...): {suma}")

---
## 🔀 Sección 2 — *Method Resolution Order* (MRO) y Herencia Múltiple

### El problema: ¿qué método llama Python cuando hay herencia múltiple?

Python usa el algoritmo **C3 Linearization** para construir el MRO — un orden total y consistente en el que se buscan los métodos. Se puede acceder con `ClaseName.__mro__` o `ClaseName.mro()`.

### Herencia simple
```text
Hijo -> Padre -> object
```

### Herencia múltiple — Problema del diamante

```text
         object
        /      \
       A        B
        \      /
           C          <- ¿C.__init__ llama a A o B primero?
```

**Regla C3:** el MRO se construye para que:
1. La clase derivada aparece antes que sus bases
2. El orden de las bases se respeta (de izquierda a derecha en `class C(A, B)`)
3. Toda clase `object` aparece al final

> `super()` + MRO es la clave para que la herencia múltiple funcione de forma predecible en Python. Es por eso que `super().__init__()` propaga correctamente la cadena de inicialización.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MRO — C3 Linearization
# ─────────────────────────────────────────────────────────────────────────────

# ── Herencia simple ──────────────────────────────────────────────────────────
print("=== Herencia simple ===")
print("ProgresionAritmetica.__mro__:")
for cls in ProgresionAritmetica.__mro__:
    print(f"  {cls.__name__}")

# ── Herencia múltiple con problema del diamante ───────────────────────────────
print("\n=== Problema del diamante ===")

class A:
    def saludar(self):
        print(f"  A.saludar (self es {type(self).__name__})")
        super().saludar()   # sin super(), B y C nunca recibirían el llamado


class B(A):
    def saludar(self):
        print(f"  B.saludar")
        super().saludar()


class C(A):
    def saludar(self):
        print(f"  C.saludar")
        super().saludar()


class D(B, C):
    def saludar(self):
        print(f"  D.saludar")
        super().saludar()


# Pero object no tiene saludar, necesitamos un terminador
class Base:
    def saludar(self):
        print(f"  Base.saludar (fin de cadena)")


class A2(Base):
    def saludar(self):
        print("  A2.saludar"); super().saludar()

class B2(A2):
    def saludar(self):
        print("  B2.saludar"); super().saludar()

class C2(A2):
    def saludar(self):
        print("  C2.saludar"); super().saludar()

class D2(B2, C2):
    def saludar(self):
        print("  D2.saludar"); super().saludar()


print("\nD2.__mro__:")
for cls in D2.__mro__:
    print(f"  {cls.__name__}")

print("\nCadena de llamadas para D2().saludar():")
D2().saludar()
# Observar: A2 sólo se llama UNA vez gracias al MRO


# ── Inspección de MRO para la jerarquía Progresion ───────────────────────────
print("\n=== MROs de la jerarquía Progresion ===")
for cls in [Progresion, ProgresionAritmetica, ProgresionGeometrica, ProgresionFibonacci]:
    mro_names = ' -> '.join(c.__name__ for c in cls.__mro__)
    print(f"  {cls.__name__:25s}: {mro_names}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Namespaces: §2.5 — dónde vive cada variable
# ─────────────────────────────────────────────────────────────────────────────

class EjemploNamespace:
    atributo_clase = "COMPARTIDO"   # vive en EjemploNamespace.__dict__

    def __init__(self, val):
        self.atributo_inst = val    # vive en self.__dict__

    def mostrar_namespaces(self):
        local_var = "local"         # vive en el frame del método

        print("=== Namespaces de búsqueda (LEGB + class) ===")
        print(f"  local_var       (local):     {local_var}")
        print(f"  self.atributo_inst (instancia): {self.atributo_inst}")
        print(f"  self.__class__.__dict__['atributo_clase'] (clase): "
              f"{self.__class__.__dict__['atributo_clase']}")


obj = EjemploNamespace("hola")
obj.mostrar_namespaces()

print("\n--- Namespace del objeto (self.__dict__) ---")
print(obj.__dict__)

print("\n--- Namespace (parcial) de la clase ---")
cls_pub = {k: v for k, v in EjemploNamespace.__dict__.items()
           if not k.startswith('__')}
print(cls_pub)

# Shadowing: la instancia puede "ocultar" un atributo de clase
print("\n--- Shadowing (instancia oculta atributo de clase) ---")
obj2 = EjemploNamespace("mundo")
print(f"Antes — obj2.atributo_clase: {obj2.atributo_clase}")
obj2.atributo_clase = "PROPIO"       # crea un atributo de instancia con mismo nombre
print(f"Después — obj2.atributo_clase: {obj2.atributo_clase}")
print(f"          EjemploNamespace.atributo_clase: {EjemploNamespace.atributo_clase}")
print(f"obj2.__dict__: {obj2.__dict__}")

---
## 🔒 Sección 3 — Encapsulamiento: Name Mangling y `@property`

### Niveles de acceso en Python

Python no tiene modificadores `private`/`protected` como Java. En cambio, usa **convenios** de nombre:

| Convención | Significado | Acceso externo |
|-----------|-------------|---------------|
| `atributo` | Público | Libre — `obj.atributo` |
| `_atributo` | "Protegido" (convención) | Técnicamente posible, desaconsejado |
| `__atributo` | "Privado" — name mangling | `obj._Clase__atributo` (contorsión) |

### Name mangling (`__atributo` con doble guión bajo)

Python transforma `__nombre` → `_NombreClase__nombre` en **tiempo de compilación**. Esto evita colisiones accidentales en herencia, no es seguridad real.

```python
class Banco:
    def __init__(self, pin):
        self.__pin = pin          # almacenado como _Banco__pin

b = Banco("1234")
# b.__pin          --> AttributeError
# b._Banco__pin    --> "1234"  (posible, pero no se debe hacer)
```

### `@property`, `@setter` y `@deleter`

```python
class Cuenta:
    @property
    def saldo(self):              # getter
        return self._saldo

    @saldo.setter
    def saldo(self, valor):       # setter con validación
        if valor < 0:
            raise ValueError("Saldo negativo")
        self._saldo = valor

    @saldo.deleter
    def saldo(self):              # deleter (raro, pero existe)
        del self._saldo
```

> **Regla de oro:** usa `@property` cuando necesitás validar, transformar o calcular el valor antes de leer/escribir. Para atributos simples sin lógica, `self.x = x` directamente es más limpio y Pythonico.

In [ ]:
class CuentaBancaria:
    """
    Cuenta bancaria con encapsulamiento robusto.
    
    Demuestra:
      - _atributo  (protegido por convención)
      - __pin      (name mangling)
      - @property  (saldo: sólo lectura)
      - @property + @setter (titular: con validación)
      - herencia correcta con super()
    """
    TASA_INTERES = 0.005    # 0.5% mensual — atributo de clase

    def __init__(self, titular: str, saldo_inicial: float, pin: str):
        self._titular = titular.strip().title()
        self.__pin    = str(pin)          # name mangling -> _CuentaBancaria__pin
        if saldo_inicial < 0:
            raise ValueError("El saldo inicial no puede ser negativo")
        self._saldo   = float(saldo_inicial)
        self._movimientos = []

    # ── Properties ──────────────────────────────────────────────────────────
    @property
    def saldo(self) -> float:
        """Saldo actual (solo lectura)."""
        return self._saldo

    @property
    def titular(self) -> str:
        return self._titular

    @titular.setter
    def titular(self, nuevo: str) -> None:
        """Permite cambiar el titular con validación."""
        nuevo = nuevo.strip()
        if not nuevo:
            raise ValueError("El titular no puede estar vacío")
        self._titular = nuevo.title()

    # ── Operaciones ─────────────────────────────────────────────────────────
    def depositar(self, monto: float, pin: str) -> None:
        self._verificar_pin(pin)
        if monto <= 0:
            raise ValueError("El depósito debe ser positivo")
        self._saldo += monto
        self._movimientos.append(('depósito',  +monto, self._saldo))

    def retirar(self, monto: float, pin: str) -> None:
        self._verificar_pin(pin)
        if monto <= 0:
            raise ValueError("El retiro debe ser positivo")
        if monto > self._saldo:
            raise ValueError(f"Saldo insuficiente (saldo: ${self._saldo:.2f})")
        self._saldo -= monto
        self._movimientos.append(('retiro', -monto, self._saldo))

    def aplicar_interes(self) -> float:
        """Aplica la tasa mensual. Retorna el monto de interés aplicado."""
        interes = self._saldo * self.TASA_INTERES
        self._saldo += interes
        self._movimientos.append(('interés', +interes, self._saldo))
        return interes

    def _verificar_pin(self, pin: str) -> None:
        """Método protegido: verifica el PIN (no expuesto como API pública)."""
        if str(pin) != self.__pin:
            raise PermissionError("PIN incorrecto")

    def extracto(self) -> str:
        lines = [f"Cuenta: {self._titular}", f"Saldo:  ${self._saldo:.2f}", "Movimientos:"]
        for tipo, monto, saldo in self._movimientos:
            signo = '+' if monto >= 0 else ''
            lines.append(f"  {tipo:10s}  {signo}${abs(monto):8.2f}  | saldo: ${saldo:.2f}")
        return '\n'.join(lines)

    def __repr__(self):
        return f"CuentaBancaria({self._titular!r}, {self._saldo:.2f})"

    def __str__(self):
        return f"Cuenta de {self._titular} | Saldo: ${self._saldo:,.2f}"


class CuentaAhorro(CuentaBancaria):
    """Subclase con tasa de interés propia y límite de retiros mensuales."""

    TASA_INTERES  = 0.012        # sobreescribe la de la clase base
    MAX_RETIROS   = 3

    def __init__(self, titular, saldo_inicial, pin):
        super().__init__(titular, saldo_inicial, pin)
        self._retiros_mes = 0

    def retirar(self, monto, pin):
        if self._retiros_mes >= self.MAX_RETIROS:
            raise PermissionError(f"Límite de {self.MAX_RETIROS} retiros mensuales alcanzado")
        super().retirar(monto, pin)
        self._retiros_mes += 1

    def nuevo_mes(self):
        self._retiros_mes = 0

    def __repr__(self):
        return f"CuentaAhorro({self._titular!r}, {self._saldo:.2f})"


# ── Demo ─────────────────────────────────────────────────────────────────────
ca = CuentaAhorro("maria garcia", 1000.0, "9876")
print(ca)
ca.depositar(500, "9876")
ca.retirar(200,  "9876")
ca.aplicar_interes()
print(ca.extracto())

# name mangling
print(f"\nAcceso directo a __pin  -> AttributeError esperado:")
try:
    _ = ca.__pin
except AttributeError as e:
    print(f"  {e}")

# Las subclases NO heredan el name mangling sin ajuste
# (cada clase tiene su propio namespace para __atributos)
print(f"\nNombre mangled en la clase: {[k for k in ca.__dict__ if 'pin' in k]}")

# PIN incorrecto
print(f"\nPIN incorrecto:")
try:
    ca.retirar(100, "0000")
except PermissionError as e:
    print(f"  {e}")

---
## 🧪 Tests Unitarios

In [ ]:
import unittest

class TestProgresiones(unittest.TestCase):

    def test_aritmetica_primeros_valores(self):
        p = ProgresionAritmetica(incremento=3, inicio=0)
        self.assertEqual(p.tomar(5), [0, 3, 6, 9, 12])

    def test_geometrica_primeros_valores(self):
        p = ProgresionGeometrica(base=2, inicio=1)
        self.assertEqual(p.tomar(6), [1, 2, 4, 8, 16, 32])

    def test_fibonacci_clasico(self):
        p = ProgresionFibonacci(0, 1)
        self.assertEqual(p.tomar(8), [0, 1, 1, 2, 3, 5, 8, 13])

    def test_fibonacci_inicio_personalizado(self):
        p = ProgresionFibonacci(4, 6)
        primeros = p.tomar(4)
        self.assertEqual(primeros, [4, 6, 10, 16])

    def test_herencia_isinstance(self):
        self.assertIsInstance(ProgresionAritmetica(), Progresion)
        self.assertIsInstance(ProgresionFibonacci(),  Progresion)

    def test_iterador_protocolo(self):
        p = ProgresionAritmetica(1, 0)
        resultado = [next(p) for _ in range(5)]
        self.assertEqual(resultado, [0, 1, 2, 3, 4])


class TestTarjetaPredatoria(unittest.TestCase):

    def setUp(self):
        self.tp = TarjetaPredatoria("Test", "Banco", "1234", 1000, 0.01)

    def test_hereda_cargar(self):
        self.assertTrue(self.tp.cargar(200))
        self.assertAlmostEqual(self.tp.saldo, 200.0)

    def test_penalidad_por_rechazo(self):
        self.tp.cargar(800)  # queda saldo=800, disponible=200
        self.assertFalse(self.tp.cargar(500))  # rechazado + penalidad de $5
        self.assertAlmostEqual(self.tp.saldo, 805.0)

    def test_cobrar_interes(self):
        self.tp.cargar(500)
        before = self.tp.saldo
        self.tp.cobrar_interes()
        self.assertAlmostEqual(self.tp.saldo, before * 1.01)

    def test_isinstance_jerarquia(self):
        self.assertIsInstance(self.tp, TarjetaCredito)
        self.assertIsInstance(self.tp, TarjetaPredatoria)


class TestCuentaBancaria(unittest.TestCase):

    def setUp(self):
        self.c = CuentaBancaria("test user", 500.0, "1234")

    def test_deposito(self):
        self.c.depositar(200, "1234")
        self.assertAlmostEqual(self.c.saldo, 700.0)

    def test_retiro(self):
        self.c.retirar(100, "1234")
        self.assertAlmostEqual(self.c.saldo, 400.0)

    def test_pin_incorrecto(self):
        with self.assertRaises(PermissionError):
            self.c.retirar(100, "0000")

    def test_saldo_readonly(self):
        with self.assertRaises(AttributeError):
            self.c.saldo = 9999

    def test_titular_setter(self):
        self.c.titular = "nuevo nombre"
        self.assertEqual(self.c.titular, "Nuevo Nombre")


# ── Run ───────────────────────────────────────────────────────────────────────
loader = unittest.TestLoader()
suite  = unittest.TestSuite()
for cls in [TestProgresiones, TestTarjetaPredatoria, TestCuentaBancaria]:
    suite.addTests(loader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exploración avanzada: MRO visual + herencia de clases built-in
# ─────────────────────────────────────────────────────────────────────────────

# ── Extender listas y diccionarios ───────────────────────────────────────────
class ListaOrdenada(list):
    """Extiende list para mantener el orden automáticamente."""

    def append(self, valor):
        super().append(valor)
        self.sort()

    def __repr__(self):
        return f"ListaOrdenada({list.__repr__(self)})"


class PilaSegura(list):
    """
    Pila (LIFO) basada en list que restringe el acceso.
    Sobreescribe append -> push, e introduce pop protegido.
    """

    def push(self, valor):
        super().append(valor)      # usa append de list

    def pop(self, *_):             # ignora índice (siempre pop del tope)
        if not self:
            raise IndexError("pop desde pila vacía")
        return super().pop()

    def peek(self):
        if not self:
            raise IndexError("peek desde pila vacía")
        return self[-1]

    # Deshabilitar acceso por índice
    def __getitem__(self, idx):
        raise TypeError("PilaSegura no soporta acceso por índice")

    def __repr__(self):
        return f"PilaSegura({list.__repr__(self)})"


# ── Demo ─────────────────────────────────────────────────────────────────────
print("=== ListaOrdenada ===")
lo = ListaOrdenada()
for v in [5, 2, 8, 1, 9, 3]:
    lo.append(v)
    print(f"  append({v}) -> {lo}")

print("\n=== PilaSegura ===")
ps = PilaSegura()
for v in [10, 20, 30]:
    ps.push(v)
print(f"  después de push(10,20,30): {list.__repr__(ps)}")  # bypass para mostrar
print(f"  peek: {ps.peek()}")
print(f"  pop:  {ps.pop()}")
print(f"  pop:  {ps.pop()}")

print("\n=== MRO ===")
print(f"ListaOrdenada: {' -> '.join(c.__name__ for c in ListaOrdenada.__mro__)}")
print(f"PilaSegura:    {' -> '.join(c.__name__ for c in PilaSegura.__mro__)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Gráfico 1: Progresiones comparadas ───────────────────────────────────────
ax = axes[0]
n_terms = 12

pa  = ProgresionAritmetica(incremento=5, inicio=0)
pg  = ProgresionGeometrica(base=1.5, inicio=1)
pfib = ProgresionFibonacci(1, 1)

terms = list(range(n_terms))
vals_a = pa.tomar(n_terms)
vals_g = pg.tomar(n_terms)
vals_f = pfib.tomar(n_terms)

ax.plot(terms, vals_a, 'b-o', label='Aritmética (Δ=5)', markersize=5)
ax.plot(terms, vals_g, 'r-s', label='Geométrica (×1.5)', markersize=5)
ax.plot(terms, vals_f, 'g-D', label='Fibonacci',         markersize=5)
ax.set_xlabel('Índice (n)')
ax.set_ylabel('Valor')
ax.set_title('Jerarquía Progresion: 3 subclases')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Gráfico 2: Diagrama de herencia %%─────────────────────────────────────────
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Jerarquía de herencia (§2.4)', fontsize=12, fontweight='bold')

def caja(ax, cx, cy, w, h, texto, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((cx - w/2, cy - h/2), w, h,
                                    boxstyle='round,pad=0.15',
                                    facecolor=color, edgecolor='#333',
                                    linewidth=1.5)
    ax.add_patch(rect)
    ax.text(cx, cy, texto, ha='center', va='center',
            fontsize=fontsize, fontweight='bold')

def flecha(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='-|>', color='#333',
                                lw=1.5, mutation_scale=15))

# object
caja(ax2, 5, 9.2, 3, 0.9, 'object', '#BDC3C7', fontsize=8)
# Progresion
caja(ax2, 5, 7.5, 3.8, 0.9, 'Progresion', '#AED6F1')
flecha(ax2, 5, 7.95, 5, 8.75)
# Subclases
for i, (cx, nombre, color) in enumerate([
        (1.8, 'Aritmetica',  '#A9DFBF'),
        (5,   'Geometrica',  '#F9E79F'),
        (8.2, 'Fibonacci',   '#FAD7A0')]):
    caja(ax2, cx, 5.5, 2.8, 0.9, nombre, color, fontsize=8)
    flecha(ax2, cx, 5.95, 5, 7.05)

# TarjetaCredito
caja(ax2, 5, 3.2, 3.8, 0.9, 'TarjetaCredito', '#D7BDE2')
flecha(ax2, 5, 3.65, 5, 4.65)   # dummy arrow spacing
ax2.text(5, 4.15, '(jerarquía separada)', ha='center', fontsize=7, style='italic', color='gray')
# TarjetaPredatoria
caja(ax2, 5, 1.8, 4, 0.9, 'TarjetaPredatoria', '#FADBD8')
flecha(ax2, 5, 2.25, 5, 2.75)

fig.tight_layout(pad=2)
plt.suptitle('Cap. 2 — §2.4 Herencia', y=1.01, fontsize=10, style='italic')
plt.savefig('herencia_diagrama.png', bbox_inches='tight', dpi=100)
plt.show()
print("Gráfico guardado.")

---
## 📋 Resumen del Notebook

### Conceptos clave del Capítulo §2.4–2.5

| Concepto | Descripción | Ejemplo |
|----------|-------------|---------|
| **Herencia** | `class Hijo(Padre)` — reutiliza y especializa | `class TarjetaPredatoria(TarjetaCredito)` |
| **`super()`** | Proxy al próximo en el MRO | `super().__init__(...)` |
| **Override** | Redefinir un método en la subclase | `def cargar(self, precio): super().cargar(...)` |
| **MRO** | Orden de búsqueda de métodos (C3) | `Clase.__mro__` |
| **Herencia múltiple** | `class C(A, B)` — requiere `super()` correcto | Problema del diamante |
| **Namespace** | Diccionario donde vive cada variable | `obj.__dict__`, `Cls.__dict__` |
| **`_atributo`** | Convención "protegido" | `self._saldo` |
| **`__atributo`** | Name mangling | `self.__pin` -> `_Clase__pin` |
| **`@property`** | Getter declarativo | `@property def saldo(self):` |
| **`@x.setter`** | Setter con validación | `@saldo.setter def saldo(self, v):` |

### Patrón Template Method (adelanto)

La jerarquía `Progresion` implementa el patrón **Template Method**:
- La clase base define el **esqueleto** del algoritmo (`__next__` llama a `_avanzar`)
- Las subclases rellenan el **detalle** sobreescribiendo `_avanzar`
- El código de iteración y despacho **no se duplica**

Este patrón se formaliza con ABCs en el Notebook 3.

---
## ✅ Autoevaluación

- [ ] Puedo crear una subclase que extiende el constructor con `super().__init__()`
- [ ] Entiendo qué hace `super()` y por qué es necesario en herencia múltiple
- [ ] Sé leer e interpretar `Clase.__mro__`
- [ ] Puedo distinguir `_nombre`, `__nombre` y `nombre` en Python
- [ ] Sé crear una property con getter y setter con validación

## 📚 Referencias y Conexiones

### Fuentes del capítulo
- Goodrich, Tamassia & Goldwasser — Cap. 2:
  - §2.4 Inheritance (Progression hierarchy, CreditCard override)
  - §2.5 Namespaces and Object-Orientation
- [Python Data Model — `super()`](https://docs.python.org/3/library/functions.html#super)
- [Python MRO — C3 linearization (PEP 3141)](https://www.python.org/download/releases/2.3/mro/)
- [Python `@property` reference](https://docs.python.org/3/library/functions.html#property)

### Continuación

| Notebook | Tema | Secciones |
|----------|------|-----------|
| **Notebook 3/3** | Polimorfismo y ADTs (ABCs) | §2.7 |
| **Ejercicios Clase 2** | Práctica: herencia + ABCs | §2.4, §2.7 |

---

© 2026 Cátedra Programación III — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
